In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/sample_data.csv")

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (100, 42)

Columns:
['Issue URL', 'ID', 'Alias', 'Classification', 'Component', 'Product', 'Version', 'Platform', 'Op sys', 'Status', 'Resolution', 'Depends on', 'Dupe of', 'Blocks', 'Groups', 'Flags', 'Severity', 'Priority', 'Deadline', 'Target Milestone', 'Creator', 'Creator Detail', 'Creation time', 'Assigned to', 'Assigned to detail', 'CC', 'CC detail', 'Is CC accessible', 'Is confirmed', 'Is open', 'Is creator accessible', 'Summary', 'Description', 'URL', 'Whiteboard', 'Keywords', 'See also', 'Last change time', 'QA contact', 'History/Activity Log', 'Comments', 'Attachments']

First 5 records:


,Issue URL,ID,Alias,Classification,Component,Product,Version,Platform,Op sys,Status,...,Description,URL,Whiteboard,Keywords,See also,Last change time,QA contact,History/Activity Log,Comments,Attachments
0,https://bugs.eclipse.org/bugs/show_bug.cgi?id=...,82174,[],Tools,cdt-other,CDT,3.0,PC,Linux,NEW,...,We need to implement automated builds for Ecli...,NaN,NaN,[],[],2020-09-04T19:17:04Z,jonah@kichwacoders.com,"[{""who"": ""jeremy@aperte.org"", ""changes"": [{""re...","{""comments"": [{""text"": ""We need to implement a...",[]
1,https://bugs.eclipse.org/bugs/show_bug.cgi?id=...,82177,[],Tools,cdt-other,CDT,3.0,All,Linux,NEW,...,"As implementation for 3.0 proceeds, we'll need...",NaN,NaN,[],[],2020-09-04T19:17:23Z,jonah@kichwacoders.com,"[{""when"": ""2005-01-04T20:18:31Z"", ""changes"": [...","{""comments"": [{""attachment_id"": null, ""id"": 38...",[]
2,https://bugs.eclipse.org/bugs/show_bug.cgi?id=...,82190,[],Tools,cdt-other,CDT,3.0,PC,Linux,NEW,...,"For 3.0 development, it would be great if we c...",NaN,NaN,[],[],2020-09-04T19:18:24Z,jonah@kichwacoders.com,"[{""who"": ""jeremy@aperte.org"", ""changes"": [{""ad...","{""comments"": [{""creation_time"": ""2005-01-04T21...",[]
3,https://bugs.eclipse.org/bugs/show_bug.cgi?id=...,102123,[],Tools,cdt-core,CDT,3.0,PC,Windows 2000,NEW,...,The GUI locks up and it uses 100% CPU.\\n\\nI ...,NaN,NaN,[],[],2020-09-04T19:21:50Z,jonah@kichwacoders.com,"[{""changes"": [{""removed"": """", ""added"": ""markus...","{""comments"": [{""time"": ""2005-06-29T05:57:57Z"",...","[{""is_obsolete"": 0, ""bug_id"": 102123, ""id"": 24..."
4,https://bugs.eclipse.org/bugs/show_bug.cgi?id=...,348709,[],Tools,terminal,CDT,Next,All,All,NEW,...,Bug 348700 fixed a severe regression late in t...,NaN,NaN,[],[],2020-09-04T19:12:17Z,jonah@kichwacoders.com,"[{""who"": ""mober.at+eclipse@gmail.com"", ""change...","{""comments"": [{""bug_id"": 348709, ""attachment_i...",[]



Missing values:
Issue URL                  0
ID                         0
Alias                      0
Classification             0
Component                  0
Product                    0
Version                    0
Platform                   0
Op sys                     0
Status                     0
Resolution               100
Depends on                 0
Dupe of                  100
Blocks                     0
Groups                     0
Flags                      0
Severity                   0
Priority                   0
Deadline                 100
Target Milestone           0
Creator                    0
Creator Detail             0
Creation time              0
Assigned to                0
Assigned to detail         0
CC                         0
CC detail                  0
Is CC accessible           0
Is confirmed               0
Is open                    0
Is creator accessible      0
Summary                    0
Description                0
URL                       

In [2]:
columns = [
    "ID",
    "Summary",
    "Description",
    "Dupe of",
    "Resolution",
    "Creation time",
    "Product",
    "Component",
]

bugs = df[columns].copy()

display(bugs.head())

print("Total reports:", len(bugs))
print("Reports with duplicate references:", bugs["Dupe of"].notna().sum())

print("\nResolution counts:")
print(bugs["Resolution"].value_counts(dropna=False))

,ID,Summary,Description,Dupe of,Resolution,Creation time,Product,Component
0,82174,Automated builds for Eclipse-RPM and Eclipse-O...,We need to implement automated builds for Ecli...,NaN,NaN,2005-01-04T20:03:58Z,CDT,cdt-other
1,82177,[RPM] Update and publish manual test plans,"As implementation for 3.0 proceeds, we'll need...",NaN,NaN,2005-01-04T20:16:59Z,CDT,cdt-other
2,82190,[RPM] Define code formatting settings for Ecli...,"For 3.0 development, it would be great if we c...",NaN,NaN,2005-01-04T21:24:51Z,CDT,cdt-other
3,102123,Cancelling C/C++ Rename refactoring brings the...,The GUI locks up and it uses 100% CPU.\\n\\nI ...,NaN,NaN,2005-06-29T05:57:57Z,CDT,cdt-core
4,348709,[terminal] Revisit closing Terminal Reader Thread,Bug 348700 fixed a severe regression late in t...,NaN,NaN,2011-06-08T11:53:33Z,CDT,terminal


Total reports: 100
Reports with duplicate references: 0

Resolution counts:
Resolution
NaN    100
Name: count, dtype: int64


In [3]:
ids = set(bugs["ID"].dropna().astype(str).str.strip())

duplicates = bugs[bugs["Dupe of"].notna()].copy()

duplicates["target_in_sample"] = (
    duplicates["Dupe of"]
    .astype(str)
    .str.strip()
    .isin(ids)
)

print("Reports marked as duplicates:", len(duplicates))
print("Duplicate targets present in sample:", duplicates["target_in_sample"].sum())

display(
    duplicates[
        ["ID", "Summary", "Dupe of", "target_in_sample"]
    ].head(10)
)

Reports marked as duplicates: 0
Duplicate targets present in sample: 0


,ID,Summary,Dupe of,target_in_sample


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

bugs["text"] = (
    bugs["Summary"].fillna("") + " " +
    bugs["Description"].fillna("")
).str.strip()

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X = vectorizer.fit_transform(bugs["text"])

print("TF-IDF matrix shape:", X.shape)

query_index = 3

similarities = cosine_similarity(
    X[query_index],
    X
).flatten()

similarities[query_index] = -1

top_indices = similarities.argsort()[-5:][::-1]

results = bugs.iloc[top_indices][["ID", "Summary"]].copy()
results["Similarity"] = similarities[top_indices]

display(results)

TF-IDF matrix shape: (100, 2283)


,ID,Summary,Similarity
91,199758,"On Windows with MinGW, if you use Managed Buil...",0.073921
27,151770,Build automatically is a really bad default,0.070607
7,352803,"Add a CDT ""change function signature"" refactor...",0.057800
17,58238,Build hangs,0.055555
13,39017,"[Managed Build] Add build/clean, etc. commands...",0.053348


In [5]:
print("QUERY REPORT")
print("ID:", bugs.iloc[query_index]["ID"])
print("Summary:", bugs.iloc[query_index]["Summary"])
print("Description:", bugs.iloc[query_index]["Description"][:500])

print("\n" + "=" * 70)

for rank, idx in enumerate(top_indices, start=1):
    report = bugs.iloc[idx]

    print(f"\nRANK {rank}")
    print("ID:", report["ID"])
    print("Summary:", report["Summary"])
    print("Similarity:", round(similarities[idx], 4))
    print("Description:", str(report["Description"])[:500])
    print("-" * 70)

QUERY REPORT
ID: 102123
Summary: Cancelling C/C++ Rename refactoring brings the machine to its knees
Description: The GUI locks up and it uses 100% CPU.\\n\\nI don't know if it ever recovers, I killed Eclipse after waiting 60-120 seconds.\\n\\nWhat information should I try to collect?\\n\\nØyvind


RANK 1
ID: 199758
Summary: On Windows with MinGW, if you use Managed Build and choose to use Parallel Build, pre-build events will not be executed and build will start from beginning
Similarity: 0.0739
Description: Build ID: I20070621-1340\\n\\nSteps To Reproduce:\\n1. Create a new MinGW C++ project on Windows with more than one source file.\\n2. Right-click the project in the project view.\\n3. Choose Properties.\\n4. In the left if the window that comes up, choose C/C++ Build.\\n5. Click the Behavior tab.\\n6. Check \\"Use Parallel Build\\".\\n7. Click \\"Use Parallel Jobs:\\".\\n8. Fill in 2, or the number of CPU's/cores you have.\\n9. Expand C/C++ Build.\\n10. Click Settings.\\n11. Click

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

summary_text = bugs["Summary"].fillna("")

summary_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_summary = summary_vectorizer.fit_transform(summary_text)

summary_scores = cosine_similarity(
    X_summary[query_index],
    X_summary
).flatten()

summary_scores[query_index] = -1

top_summary_indices = summary_scores.argsort()[-5:][::-1]

summary_results = bugs.iloc[top_summary_indices][
    ["ID", "Summary"]
].copy()

summary_results["Similarity"] = summary_scores[top_summary_indices]

display(summary_results)

,ID,Summary,Similarity
54,174054,API for retrieving concrete project types that...,0.135268
7,352803,"Add a CDT ""change function signature"" refactor...",0.134185
99,205113,No build if file changed outside Eclipse,0.000000
98,204988,CDT can't build with Microsoft SDK for the Sub...,0.000000
95,202841,CDT lets a user choose a tool for a header file,0.000000


In [8]:
import numpy as np

valid_indices = np.where(summary_scores > 0)[0]

ranked_indices = valid_indices[
    np.argsort(summary_scores[valid_indices])[::-1]
]

top_indices = ranked_indices[:5]

results = bugs.iloc[top_indices][["ID", "Summary"]].copy()
results["Similarity"] = summary_scores[top_indices]

display(results)

,ID,Summary,Similarity
54,174054,API for retrieving concrete project types that...,0.135268
7,352803,"Add a CDT ""change function signature"" refactor...",0.134185
